In [25]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [26]:
train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

train_df.shape

(2000, 8)

In [27]:
label2id = {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
    "E": 4
}

train_df["label"] = train_df["answer"].map(label2id)

print(train_df.loc[150, ["answer", "label"]])

answer    C
label     2
Name: 150, dtype: object


In [28]:
row = train_df.loc[0]

option_b_input = str(row["prompt"]) + " [SEP] " + str(row["B"])

print(option_b_input)
print("Character length:", len(option_b_input))

Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options. [SEP] Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.
Character length: 407


In [29]:
from transformers import AutoTokenizer, AutoModelForMultipleChoice
import torch

model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForMultipleChoice.from_pretrained(model_name)

row = train_df.loc[0]

first_sentences = [str(row["prompt"])] * 5

second_sentences = [
    str(row["A"]),
    str(row["B"]),
    str(row["C"]),
    str(row["D"]),
    str(row["E"])
]

encoding = tokenizer(
    first_sentences,
    second_sentences,
    truncation=True,
    padding="max_length",
    max_length=128,
    return_tensors="pt"
)

encoding["input_ids"].shape

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


torch.Size([5, 128])

In [30]:
input_ids = encoding["input_ids"].unsqueeze(0)
attention_mask = encoding["attention_mask"].unsqueeze(0)

print(input_ids.shape)

label = torch.tensor([row["label"]])

outputs = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=label
)

print(outputs.loss.shape)

torch.Size([1, 5, 128])
torch.Size([])


In [31]:
!pip install -q -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 29.0 MB/s eta 0:00:0000:0100:01


In [32]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

model = get_peft_model(model, lora_config)

trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print("Trainable parameters:", trainable_params)

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cpu).


Trainable parameters: 295681


In [33]:
from datasets import Dataset

train_100 = train_df.iloc[:100].copy()

hf_dataset = Dataset.from_pandas(train_100)

print(hf_dataset)

Dataset({
    features: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer', 'label'],
    num_rows: 100
})


In [34]:
def preprocess_function(example):
    
    first_sentences = [str(example["prompt"])] * 5

    second_sentences = [
        str(example["A"]),
        str(example["B"]),
        str(example["C"]),
        str(example["D"]),
        str(example["E"])
    ]

    encoding = tokenizer(
        first_sentences,
        second_sentences,
        truncation=True,
        padding="max_length",
        max_length=128,
    )

    encoding["labels"] = example["label"]

    return encoding

In [36]:
tokenized_dataset = hf_dataset.map(preprocess_function)

tokenized_dataset = tokenized_dataset.remove_columns(
    ["id", "prompt", "A", "B", "C", "D", "E", "answer"]
)

tokenized_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

sample = tokenized_dataset[0]

print(sample.keys())
print(sample["input_ids"].shape)

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

dict_keys(['input_ids', 'attention_mask', 'labels'])
torch.Size([5, 128])


In [38]:
train_32 = train_df.iloc[:32].copy()

hf_dataset_32 = Dataset.from_pandas(train_32)

def preprocess_function_64(example):

    first_sentences = [str(example["prompt"])] * 5

    second_sentences = [
        str(example["A"]),
        str(example["B"]),
        str(example["C"]),
        str(example["D"]),
        str(example["E"])
    ]

    encoding = tokenizer(
        first_sentences,
        second_sentences,
        truncation=True,
        padding="max_length",
        max_length=64,
    )

    encoding["labels"] = example["label"]

    return encoding

In [40]:
from transformers import TrainingArguments, Trainer

tokenized_dataset_32 = hf_dataset_32.map(preprocess_function_64)

tokenized_dataset_32 = tokenized_dataset_32.remove_columns(
    ["id", "prompt", "A", "B", "C", "D", "E", "answer"]
)

tokenized_dataset_32.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

training_args = TrainingArguments(
    output_dir="./mcq_lora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    logging_steps=1,
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset_32,
)

train_result = trainer.train()

print("Global Step:", trainer.state.global_step)

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,1.682988
2,1.590020
3,1.637105
4,1.683388


Global Step: 4


In [41]:
row = train_df.loc[0]

first_sentences = [str(row["prompt"])] * 5

second_sentences = [
    str(row["A"]),
    str(row["B"]),
    str(row["C"]),
    str(row["D"]),
    str(row["E"])
]

encoding = tokenizer(
    first_sentences,
    second_sentences,
    truncation=True,
    padding="max_length",
    max_length=64,
    return_tensors="pt"
)

input_ids = encoding["input_ids"].unsqueeze(0)
attention_mask = encoding["attention_mask"].unsqueeze(0)

model.eval()

with torch.no_grad():
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

probabilities = torch.softmax(outputs.logits, dim=1)

print(probabilities)

tensor([[0.2057, 0.2011, 0.1953, 0.1961, 0.2019]])
